# Factory Pipeline Demo: Config-Driven Recommender Construction

This notebook demonstrates `create_recommender_pipeline` — the single entry point for
building any recommender pipeline scikit-rec supports, purely from a config dict.

**What it covers:**
- Tabular XGBoost + UniversalScorer + RankingRecommender (train + recommend)
- Embedding (MatrixFactorization) + UniversalScorer + RankingRecommender (train + recommend)
- Config validation: what happens with mismatched configs

**Data:** Uses the `sample_binary_reward` dataset shipped with the library (5,000 interactions, 3 items).

## 1. Setup

In [1]:
import logging
from pathlib import Path

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.dataset.users_dataset import UsersDataset
from skrec.orchestrator import create_recommender_pipeline

logging.basicConfig(level=logging.WARNING)

# Load sample data shipped with the library
DATA_DIR = Path("../skrec/examples/datasets/sample_binary_reward")

interactions_ds = InteractionsDataset(data_location=str(DATA_DIR / "interactions.csv"))
items_ds = ItemsDataset(data_location=str(DATA_DIR / "items.csv"))
users_ds = UsersDataset(data_location=str(DATA_DIR / "users.csv"))

print("Datasets created from sample_binary_reward:")
print(f"  Interactions: {DATA_DIR / 'interactions.csv'}")
print(f"  Items:        {DATA_DIR / 'items.csv'}")
print(f"  Users:        {DATA_DIR / 'users.csv'}")

Datasets created from sample_binary_reward:
  Interactions: ../skrec/examples/datasets/sample_binary_reward/interactions.csv
  Items:        ../skrec/examples/datasets/sample_binary_reward/items.csv
  Users:        ../skrec/examples/datasets/sample_binary_reward/users.csv


## 2. Tabular Pipeline: XGBoost + Universal + Ranking

The simplest config — a tabular XGBoost classifier with a universal scorer and ranking recommender. This is the default path that existed before the factory extension.

In [2]:
tabular_config = {
    "recommender_type": "ranking",
    "scorer_type": "universal",
    "estimator_config": {
        "ml_task": "classification",
        "xgboost": {
            "n_estimators": 50,
            "max_depth": 3,
            "learning_rate": 0.1,
        },
    },
}

tabular_pipeline = create_recommender_pipeline(tabular_config)
print(f"Pipeline type:   {type(tabular_pipeline).__name__}")
print(f"Scorer type:     {type(tabular_pipeline.scorer).__name__}")
print(f"Estimator type:  {type(tabular_pipeline.scorer.estimator).__name__}")

2026-04-17 15:51:51,701 - skrec.orchestrator.factory - INFO Creating recommender pipeline from config...


INFO:skrec.orchestrator.factory:Creating recommender pipeline from config...


2026-04-17 15:51:51,701 - skrec.orchestrator.factory - INFO Creating estimator. Estimator type: tabular


INFO:skrec.orchestrator.factory:Creating estimator. Estimator type: tabular


2026-04-17 15:51:51,702 - skrec.orchestrator.factory - INFO Creating estimator. ML Task: classification, Scorer Type Hint: universal, Tuned Mode: False


...

2026-04-17 15:51:51,703 - skrec.orchestrator.factory - INFO Creating recommender of type: ranking


INFO:skrec.orchestrator.factory:Creating recommender of type: ranking


2026-04-17 15:51:51,703 - skrec.orchestrator.factory - INFO Recommender pipeline created successfully.


INFO:skrec.orchestrator.factory:Recommender pipeline created successfully.


Pipeline type:   RankingRecommender
Scorer type:     TabularUniversalScorer
Estimator type:  XGBClassifierEstimator


In [3]:
# Train the pipeline
tabular_pipeline.train(
    interactions_ds=interactions_ds,
    items_ds=items_ds,
    users_ds=users_ds,
)
print("Training complete.")
print(f"Features learned: {tabular_pipeline.scorer.estimator.feature_names}")

Training complete.
Features learned: ['feat1', 'feat2', 'item_feat1', 'item_feat2', 'item_feat3', 'item_feat4', 'item_feat5', 'item_feat6', 'item_feat7']


In [4]:
import pandas as pd

# Recommend top-2 items for a few users
# The recommend() call needs user features. We pass the users DataFrame
# so the scorer can look up features by USER_ID.
users_df = pd.read_csv(DATA_DIR / "users.csv")
query = users_df.head(5)
recs = tabular_pipeline.recommend(interactions=query, top_k=2)

for i, user_id in enumerate(query["USER_ID"]):
    print(f"  {user_id}: {list(recs[i])}")

2026-04-17 15:51:51,786 - skrec.scorer.base_scorer - INFO Receiving DataFrames for Interactions and Users


INFO:skrec.scorer.base_scorer:Receiving DataFrames for Interactions and Users


2026-04-17 15:51:51,786 - skrec.scorer.base_scorer - INFO Shape of Interactions DataFrame: (5, 3)


INFO:skrec.scorer.base_scorer:Shape of Interactions DataFrame: (5, 3)


2026-04-17 15:51:51,786 - skrec.scorer.base_scorer - INFO Shape of Users DataFrame: (5, 1)


...

2026-04-17 15:51:51,788 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


INFO:skrec.scorer.universal:Adding Item Features for All Items, via Replication


2026-04-17 15:51:51,789 - skrec.scorer.universal - INFO Completed Adding Item Features for ALL ITEMS, via Replication


INFO:skrec.scorer.universal:Completed Adding Item Features for ALL ITEMS, via Replication


  user_0: ['ITEM_3', 'ITEM_2']
  user_1: ['ITEM_1', 'ITEM_2']
  user_2: ['ITEM_3', 'ITEM_2']
  user_3: ['ITEM_1', 'ITEM_2']
  user_4: ['ITEM_3', 'ITEM_1']


## 3. Embedding Pipeline: Matrix Factorization + Universal + Ranking

Same scorer and recommender, but swapping the estimator to a collaborative filtering model. The only config change is `estimator_type: "embedding"` with the `embedding` section.

In [5]:
embedding_config = {
    "recommender_type": "ranking",
    "scorer_type": "universal",
    "estimator_config": {
        "estimator_type": "embedding",
        "embedding": {
            "model_type": "matrix_factorization",
            "params": {
                "n_factors": 16,
                "epochs": 20,
                "learning_rate": 0.01,
            },
        },
    },
}

embedding_pipeline = create_recommender_pipeline(embedding_config)
print(f"Pipeline type:   {type(embedding_pipeline).__name__}")
print(f"Scorer type:     {type(embedding_pipeline.scorer).__name__}")
print(f"Estimator type:  {type(embedding_pipeline.scorer.estimator).__name__}")

2026-04-17 15:51:51,794 - skrec.orchestrator.factory - INFO Creating recommender pipeline from config...


INFO:skrec.orchestrator.factory:Creating recommender pipeline from config...


2026-04-17 15:51:51,794 - skrec.orchestrator.factory - INFO Creating estimator. Estimator type: embedding


INFO:skrec.orchestrator.factory:Creating estimator. Estimator type: embedding


2026-04-17 15:51:51,795 - skrec.orchestrator.factory - INFO Creating MatrixFactorizationEstimator with params: {'n_factors': 16, 'epochs': 20, 'learning_rate': 0.01}


...

2026-04-17 15:51:51,796 - skrec.orchestrator.factory - INFO Creating recommender of type: ranking


INFO:skrec.orchestrator.factory:Creating recommender of type: ranking


2026-04-17 15:51:51,796 - skrec.orchestrator.factory - INFO Recommender pipeline created successfully.


INFO:skrec.orchestrator.factory:Recommender pipeline created successfully.


Pipeline type:   RankingRecommender
Scorer type:     EmbeddingUniversalScorer
Estimator type:  MatrixFactorizationEstimator


In [6]:
# Train and recommend
embedding_pipeline.train(
    interactions_ds=interactions_ds,
    items_ds=items_ds,
    users_ds=users_ds,
)
print("Training complete.")

# Embedding models use USER_ID to look up learned embeddings
query_emb = pd.DataFrame({"USER_ID": ["user_0", "user_1", "user_2", "user_3", "user_4"]})
recs = embedding_pipeline.recommend(interactions=query_emb, top_k=2)
for i, user_id in enumerate(query_emb["USER_ID"]):
    print(f"  {user_id}: {list(recs[i])}")

2026-04-17 15:51:51,807 - skrec.estimator.embedding.matrix_factorization_estimator - WARNING Customer (user) features are not supported in this basic collaborative filtering setup. Only user IDs are used; extra columns in the users DataFrame are ignored.


2026-04-17 15:51:51,808 - skrec.estimator.embedding.matrix_factorization_estimator - WARNING Item features are not supported in this basic collaborative filtering setup. Only item IDs are used; extra columns in the items DataFrame are ignored.


2026-04-17 15:51:53,566 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


INFO:skrec.scorer.universal:Adding Item Features for All Items, via Replication


Training complete.
  user_0: ['ITEM_2', 'ITEM_1']
  user_1: ['ITEM_3', 'ITEM_1']
  user_2: ['ITEM_2', 'ITEM_1']
  user_3: ['ITEM_1', 'ITEM_3']
  user_4: ['ITEM_1', 'ITEM_3']


## 4. Config Validation: What Happens with Bad Configs

The factory catches mismatches early with clear error messages. Here are a few examples.

In [7]:
# Example 1: Embedding estimator with incompatible scorer
try:
    create_recommender_pipeline({
        "recommender_type": "ranking",
        "scorer_type": "independent",  # independent doesn't support embeddings
        "estimator_config": {
            "estimator_type": "embedding",
            "embedding": {"model_type": "ncf"},
        },
    })
except ValueError as e:
    print(f"Caught: {e}")

# Example 2: Sequential recommender without sequential estimator
try:
    create_recommender_pipeline({
        "recommender_type": "sequential",
        "scorer_type": "sequential",
        "estimator_config": {"ml_task": "classification"},  # tabular, not sequential
    })
except ValueError as e:
    print(f"Caught: {e}")

# Example 3: Unknown recommender type (typo)
try:
    create_recommender_pipeline({
        "recommender_type": "ranknig",  # typo
        "scorer_type": "universal",
        "estimator_config": {"ml_task": "classification"},
    })
except NotImplementedError as e:
    print(f"Caught: {e}")

2026-04-17 15:51:53,572 - skrec.orchestrator.factory - INFO Creating recommender pipeline from config...


INFO:skrec.orchestrator.factory:Creating recommender pipeline from config...


2026-04-17 15:51:53,573 - skrec.orchestrator.factory - INFO Creating recommender pipeline from config...


INFO:skrec.orchestrator.factory:Creating recommender pipeline from config...


2026-04-17 15:51:53,573 - skrec.orchestrator.factory - INFO Creating recommender pipeline from config...


...

2026-04-17 15:51:53,575 - skrec.orchestrator.factory - INFO Creating scorer of type: universal


INFO:skrec.orchestrator.factory:Creating scorer of type: universal


2026-04-17 15:51:53,576 - skrec.orchestrator.factory - INFO Creating recommender of type: ranknig


INFO:skrec.orchestrator.factory:Creating recommender of type: ranknig


Caught: scorer_type 'independent' does not support embedding estimators. Use scorer_type='universal' with embedding estimators.
Caught: recommender_type 'sequential' requires estimator_type 'sequential', got 'tabular'.
Caught: Recommender type 'ranknig' not supported. Supported types: 'ranking', 'bandits', 'sequential', 'hierarchical_sequential', 'uplift', 'gcsl'.
